<a href="https://www.kaggle.com/code/obaidah/asr-trainning?scriptVersionId=311093952" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [4]:
# 0) INSTALL / IMPORTS
!pip -q install librosa soundfile

import os
import ast
import json
import math
import time
import shutil
import random
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [5]:
# 2) PATHS
INPUT_BASE = "/kaggle/input/datasets/obaidah"
WORK_BASE = "/kaggle/working"

CHECKPOINTS_SRC = os.path.join(INPUT_BASE, "checkpoints", "ASR_checkpoints")
DATA_SRC = os.path.join(INPUT_BASE, "asr-nn")
AUDIO_DIR = "/kaggle/working/waves"   

CHECKPOINTS_DST = os.path.join(WORK_BASE, "checkpoints")
DATA_DST = os.path.join(WORK_BASE, "data")

os.makedirs(CHECKPOINTS_DST, exist_ok=True)
os.makedirs(DATA_DST, exist_ok=True)

In [6]:
# 1) BASIC CONFIG
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = torch.cuda.is_available()
DEVICE_TYPE = "cuda" if USE_CUDA else "cpu"

print("=" * 70)
print("Device:", DEVICE)
print("CUDA available:", USE_CUDA)
print("=" * 70)

Device: cuda
CUDA available: True


In [7]:
# 3) COPY FILES (NO WAVES COPY)
checkpoint_files = [
    "last_checkpoint.pt",
    "best_model.pt",
    "history.json",
    "training_state.json",
    "char2idx.json",
    "idx2char.json",
    "train_ready.csv",
    "val_ready.csv",
    "test_ready.csv",
]

data_files = [
    "train.csv",
    "val.csv",
    "test.csv",
    "resample_success.csv",
    "resample_errors.csv",
]

print("\n[1/8] Copying small files to /kaggle/working ...")

for fname in tqdm(checkpoint_files, desc="Copy checkpoint files"):
    src = os.path.join(CHECKPOINTS_SRC, fname)
    dst = os.path.join(CHECKPOINTS_DST, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)

for fname in tqdm(data_files, desc="Copy csv files"):
    src = os.path.join(DATA_SRC, fname)
    dst = os.path.join(DATA_DST, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)

print(" Copy step done.")



[1/8] Copying small files to /kaggle/working ...


Copy checkpoint files:   0%|          | 0/9 [00:00<?, ?it/s]

Copy csv files:   0%|          | 0/5 [00:00<?, ?it/s]

 Copy step done.


In [8]:
# 4) DEFINITIONS OF LOCAL FILES
checkpoint_dir = CHECKPOINTS_DST

last_ckpt_path = os.path.join(checkpoint_dir, "last_checkpoint.pt")
best_ckpt_path = os.path.join(checkpoint_dir, "best_model.pt")
history_path = os.path.join(checkpoint_dir, "history.json")
training_state_path = os.path.join(checkpoint_dir, "training_state.json")

train_ready_path = os.path.join(checkpoint_dir, "train_ready.csv")
val_ready_path   = os.path.join(checkpoint_dir, "val_ready.csv")
test_ready_path  = os.path.join(checkpoint_dir, "test_ready.csv")

char2idx_path = os.path.join(checkpoint_dir, "char2idx.json")
idx2char_path = os.path.join(checkpoint_dir, "idx2char.json")

print("\n[2/8] Checking important files ...")
important_paths = {
    "AUDIO_DIR": AUDIO_DIR,
    "last_checkpoint.pt": last_ckpt_path,
    "best_model.pt": best_ckpt_path,
    "history.json": history_path,
    "training_state.json": training_state_path,
    "train_ready.csv": train_ready_path,
    "val_ready.csv": val_ready_path,
    "test_ready.csv": test_ready_path,
    "char2idx.json": char2idx_path,
    "idx2char.json": idx2char_path,
}

for name, path in important_paths.items():
    print(f"{name:<22} -> {'✅' if os.path.exists(path) else '❌'} {path}")


[2/8] Checking important files ...
AUDIO_DIR              -> ✅ /kaggle/working/waves
last_checkpoint.pt     -> ✅ /kaggle/working/checkpoints/last_checkpoint.pt
best_model.pt          -> ✅ /kaggle/working/checkpoints/best_model.pt
history.json           -> ✅ /kaggle/working/checkpoints/history.json
training_state.json    -> ✅ /kaggle/working/checkpoints/training_state.json
train_ready.csv        -> ✅ /kaggle/working/checkpoints/train_ready.csv
val_ready.csv          -> ✅ /kaggle/working/checkpoints/val_ready.csv
test_ready.csv         -> ✅ /kaggle/working/checkpoints/test_ready.csv
char2idx.json          -> ✅ /kaggle/working/checkpoints/char2idx.json
idx2char.json          -> ✅ /kaggle/working/checkpoints/idx2char.json


In [9]:
# 5) LOAD CSV + VOCAB
print("\n[3/8] Loading ready CSV files and vocab ...")

train_df = pd.read_csv(train_ready_path)
val_df   = pd.read_csv(val_ready_path)
test_df  = pd.read_csv(test_ready_path)

train_df["resampled_path"] = train_df["resampled_path"].apply(
    lambda x: os.path.join(AUDIO_DIR, os.path.basename(x))
)

val_df["resampled_path"] = val_df["resampled_path"].apply(
    lambda x: os.path.join(AUDIO_DIR, os.path.basename(x))
)

test_df["resampled_path"] = test_df["resampled_path"].apply(
    lambda x: os.path.join(AUDIO_DIR, os.path.basename(x))
)

with open(char2idx_path, "r", encoding="utf-8") as f:
    char2idx = json.load(f)

with open(idx2char_path, "r", encoding="utf-8") as f:
    idx2char = json.load(f)

# idx2char keys are strings in json
idx2char = {int(k): v for k, v in idx2char.items()}

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)
print("Vocab size :", len(char2idx))



[3/8] Loading ready CSV files and vocab ...
Train shape: (62973, 7)
Val shape  : (7872, 7)
Test shape : (7872, 7)
Vocab size : 32


In [10]:
# 6) HELPER FUNCTIONS
def parse_label_ids(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return ast.literal_eval(x)

for df in [train_df, val_df, test_df]:
    df["label_ids"] = df["label_ids"].apply(parse_label_ids)

# لو resampled_path مش موجود أو path قديم من colab
def fix_paths(df):
    # لو المسار already صحيح → سيبه
    if df["resampled_path"].iloc[0].startswith("/kaggle"):
        print("✅ Paths already correct")
        return df
    
    print("⚠️ Fixing paths...")

    df["resampled_path"] = df["resampled_path"].apply(
        lambda x: os.path.join(AUDIO_DIR, os.path.basename(x))
    )
    
    return df

train_df = fix_paths(train_df)
val_df   = fix_paths(val_df)
test_df  = fix_paths(test_df)

# فلترة أي ملفات ناقصة
print("\nChecking audio existence ...")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    exists_mask = df["resampled_path"].apply(os.path.exists)
    print(f"{name}: {exists_mask.sum()}/{len(df)} audio files found")
    
train_df = train_df[train_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)
val_df   = val_df[val_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)
test_df  = test_df[test_df["resampled_path"].apply(os.path.exists)].reset_index(drop=True)

print("After filtering missing files:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

✅ Paths already correct
✅ Paths already correct
✅ Paths already correct

Checking audio existence ...
Train: 62973/62973 audio files found
Val: 7872/7872 audio files found
Test: 7872/7872 audio files found
After filtering missing files:
Train: (62973, 7)
Val  : (7872, 7)
Test : (7872, 7)


In [11]:
# 7) DATASET + COLLATE
TARGET_SR = 16000
N_MELS = 80
N_FFT = 400
HOP_LENGTH = 160
WIN_LENGTH = 400

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    win_length=WIN_LENGTH,
    n_mels=N_MELS,
    center=True,
    power=2.0
)

amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)

class ArabicASRDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio_path = row["resampled_path"]
        text = row["normalized_text"]
        label_ids = row["label_ids"]

        waveform, sr = torchaudio.load(audio_path)  # [C, T]

        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        if sr != TARGET_SR:
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)

        waveform = waveform.squeeze(0)  # [T]

        features = mel_transform(waveform)          # [n_mels, time]
        features = amplitude_to_db(features)        # [n_mels, time]
        features = features.transpose(0, 1)         # [time, n_mels]


        labels = torch.tensor(label_ids, dtype=torch.long)

        return {
            "features": features,
            "feature_length": features.size(0),
            "labels": labels,
            "label_length": len(labels),
            "text": text,
            "audio_path": audio_path
        }

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    features = [b["features"] for b in batch]
    labels = [b["labels"] for b in batch]

    feature_lengths = torch.tensor([b["feature_length"] for b in batch], dtype=torch.long)
    label_lengths = torch.tensor([b["label_length"] for b in batch], dtype=torch.long)

    padded_features = nn.utils.rnn.pad_sequence(features, batch_first=True, padding_value=0.0)
    concatenated_labels = torch.cat(labels, dim=0)

    texts = [b["text"] for b in batch]
    audio_paths = [b["audio_path"] for b in batch]

    return {
        "features": padded_features,
        "feature_lengths": feature_lengths,
        "labels": concatenated_labels,
        "label_lengths": label_lengths,
        "texts": texts,
        "audio_paths": audio_paths
    }

print("\n[4/8] Building datasets and dataloaders ...")

BATCH_SIZE = 32
NUM_WORKERS = 4

train_dataset = ArabicASRDataset(train_df)
val_dataset   = ArabicASRDataset(val_df)
test_dataset  = ArabicASRDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

print(f"BATCH_SIZE = {BATCH_SIZE}")
print(f"NUM_WORKERS = {NUM_WORKERS}")

sample_batch = next(iter(train_loader))
print("features shape       :", sample_batch["features"].shape)
print("feature_lengths shape:", sample_batch["feature_lengths"].shape)
print("labels shape         :", sample_batch["labels"].shape)
print("label_lengths shape  :", sample_batch["label_lengths"].shape)
print("sample text          :", sample_batch["texts"][0])
print("sample audio path    :", sample_batch["audio_paths"][0])

sample_batch = next(iter(train_loader))
print(sample_batch["audio_paths"][0])


[4/8] Building datasets and dataloaders ...
BATCH_SIZE = 32
NUM_WORKERS = 4
features shape       : torch.Size([32, 730, 80])
feature_lengths shape: torch.Size([32])
labels shape         : torch.Size([784])
label_lengths shape  : torch.Size([32])
sample text          : لدي صديق يعيش في سابورو
sample audio path    : /kaggle/working/waves/common_voice_ar_21262281.wav
/kaggle/working/waves/common_voice_ar_19478824.wav


In [12]:
# 8) MODEL
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes, n_mels=80, proj_dim=256, lstm_hidden=256, lstm_layers=2, dropout=0.2):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
            nn.Dropout(0.1),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
            nn.Dropout(0.1),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Dropout(dropout),
        )

        cnn_out_dim = 128 * 20

        self.projection = nn.Sequential(
            nn.Linear(cnn_out_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=proj_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x, input_lengths):
        x = x.unsqueeze(1)   # [B, 1, T, n_mels]
        x = self.cnn(x)      # [B, C, T', F']

        b, c, t, f = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(b, t, c * f)

        x = self.projection(x)

        output_lengths = input_lengths.clone()
        output_lengths = torch.div(output_lengths, 2, rounding_mode='floor')
        output_lengths = torch.div(output_lengths, 2, rounding_mode='floor')
        output_lengths = torch.div(output_lengths, 2, rounding_mode='floor')
        output_lengths = torch.clamp(output_lengths, min=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths=output_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)

        x, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        logits = self.classifier(x)
        log_probs = F.log_softmax(logits, dim=-1)

        return log_probs, output_lengths


print("\n[5/8] Building model ...")
num_classes = len(char2idx)

model = CNNBiLSTMCTC(
    num_classes=num_classes,
    n_mels=N_MELS,
    proj_dim=256,
    lstm_hidden=256,
    lstm_layers=2,
    dropout=0.2
)

if torch.cuda.is_available():
    print("CUDA device count available:", torch.cuda.device_count())
    print("✅ Using SINGLE GPU for maximum stability")
    DEVICE = torch.device("cuda:0")
    model = model.to(DEVICE)
    USE_CUDA = True
    DEVICE_TYPE = "cuda"
else:
    DEVICE = torch.device("cpu")
    model = model.to(DEVICE)
    USE_CUDA = False
    DEVICE_TYPE = "cpu"

print(model)
print("Model device:", DEVICE)
print("Total parameters:", sum(p.numel() for p in model.parameters()))


[5/8] Building model ...
CUDA device count available: 2
✅ Using SINGLE GPU for maximum stability
CNNBiLSTMCTC(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.1, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stat

In [13]:
# 9) TRAINING CONFIG
EPOCHS_PER_RUN = 3
PATIENCE = 3
MIN_DELTA = 0.001

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 5.0

criterion = nn.CTCLoss(blank=0, zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(DEVICE_TYPE, enabled=USE_CUDA)

best_val_loss = float("inf")
start_epoch = 0
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "val_loss": []
}


In [14]:
# 10) SAVE / LOAD HELPERS
def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def save_checkpoint(path, epoch, best_val_loss, epochs_without_improvement, history):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history
    }, path)

def load_checkpoint_if_exists():
    global best_val_loss, start_epoch, epochs_without_improvement, history

    if os.path.exists(last_ckpt_path):
        print(f"\n✅ Found checkpoint: {last_ckpt_path}")
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE)

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch = ckpt.get("epoch", -1) + 1
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        epochs_without_improvement = ckpt.get("epochs_without_improvement", 0)
        history = ckpt.get("history", {"train_loss": [], "val_loss": []})

        print(f"Resumed from epoch {start_epoch}")
        print(f"Best val loss so far: {best_val_loss:.4f}")
        print(f"epochs_without_improvement: {epochs_without_improvement}")
        print(f"History length: train={len(history['train_loss'])}, val={len(history['val_loss'])}")
    else:
        print("\n⚠️ No checkpoint found. Starting from scratch.")

In [15]:
# 11) TRAIN / VALIDATE
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        features = batch["features"].to(device, non_blocking=True)
        feature_lengths = batch["feature_lengths"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        label_lengths = batch["label_lengths"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        try:
            with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
                log_probs, output_lengths = model(features, feature_lengths)
                log_probs = log_probs.permute(1, 0, 2)   # [T, B, C]

                loss = criterion(
                    log_probs,
                    labels,
                    output_lengths,
                    label_lengths
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            processed_batches += 1

            avg_loss = running_loss / processed_batches
            pbar.set_postfix({
                "train_loss": f"{avg_loss:.4f}",
                "batch_loss": f"{loss.item():.4f}"
            })

        except RuntimeError as e:
            raise RuntimeError(f"Fatal error at train batch {batch_idx}: {e}")

    if processed_batches == 0:
        return float("inf")

    return running_loss / processed_batches


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    processed_batches = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        features = batch["features"].to(device, non_blocking=True)
        feature_lengths = batch["feature_lengths"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        label_lengths = batch["label_lengths"].to(device, non_blocking=True)

        try:
            with torch.amp.autocast(device_type=DEVICE_TYPE, enabled=USE_CUDA):
                log_probs, output_lengths = model(features, feature_lengths)
                log_probs = log_probs.permute(1, 0, 2)   # [T, B, C]

                loss = criterion(
                    log_probs,
                    labels,
                    output_lengths,
                    label_lengths
                )

            running_loss += loss.item()
            processed_batches += 1

            avg_loss = running_loss / processed_batches
            pbar.set_postfix({
                "val_loss": f"{avg_loss:.4f}",
                "batch_loss": f"{loss.item():.4f}"
            })

        except RuntimeError as e:
            raise RuntimeError(f"Fatal error at val batch {batch_idx}: {e}")

    if processed_batches == 0:
        return float("inf")

    return running_loss / processed_batches

In [16]:
# 12) RESUME
print("\n[6/8] Loading checkpoint state ...")
load_checkpoint_if_exists()

end_epoch = start_epoch + EPOCHS_PER_RUN
print(f"Will train from epoch {start_epoch} to {end_epoch - 1}")


[6/8] Loading checkpoint state ...

✅ Found checkpoint: /kaggle/working/checkpoints/last_checkpoint.pt
Resumed from epoch 14
Best val loss so far: 0.7231
epochs_without_improvement: 0
History length: train=14, val=14
Will train from epoch 14 to 16


In [17]:
# 13) TRAIN LOOP
print("\n[7/8] Starting training ...")

for epoch in range(start_epoch, end_epoch):
    print(f"\n{'=' * 70}")
    print(f"Epoch {epoch + 1}")
    print(f"{'=' * 70}")

    epoch_start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, DEVICE)
    val_loss = validate_one_epoch(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    epoch_time_min = (time.time() - epoch_start_time) / 60.0
    improvement = best_val_loss - val_loss if best_val_loss != float("inf") else 0.0
    generalization_gap = val_loss - train_loss

    print(f"Epoch {epoch + 1}:")
    print(f"  train_loss = {train_loss:.4f}")
    print(f"  val_loss   = {val_loss:.4f}")
    print(f"  improvement = {improvement:.6f}")
    print(f"  generalization_gap = {generalization_gap:.4f}")
    print(f"  time = {epoch_time_min:.2f} min")

    # save last checkpoint every epoch
    save_checkpoint(
        path=last_ckpt_path,
        epoch=epoch,
        best_val_loss=best_val_loss,
        epochs_without_improvement=epochs_without_improvement,
        history=history
    )

    save_json(history_path, history)

    save_json(training_state_path, {
        "last_completed_epoch": epoch,
        "next_start_epoch": epoch + 1,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "epochs_per_run": EPOCHS_PER_RUN,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "last_train_loss": train_loss,
        "last_val_loss": val_loss,
        "last_improvement": improvement,
        "last_generalization_gap": generalization_gap,
        "last_epoch_time_min": epoch_time_min
    })

    # check improvement
    if val_loss < (best_val_loss - MIN_DELTA):
        best_val_loss = val_loss
        epochs_without_improvement = 0

        save_checkpoint(
            path=best_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        save_checkpoint(
            path=last_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        save_json(training_state_path, {
            "last_completed_epoch": epoch,
            "next_start_epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "epochs_without_improvement": epochs_without_improvement,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "epochs_per_run": EPOCHS_PER_RUN,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "last_train_loss": train_loss,
            "last_val_loss": val_loss,
            "last_improvement": improvement,
            "last_generalization_gap": generalization_gap,
            "last_epoch_time_min": epoch_time_min
        })

        print(f"✅ Saved BEST model at epoch {epoch + 1} with val_loss = {val_loss:.4f}")

    else:
        epochs_without_improvement += 1
        print(f"⚠️ No significant improvement. Counter: {epochs_without_improvement}/{PATIENCE}")

        save_checkpoint(
            path=last_ckpt_path,
            epoch=epoch,
            best_val_loss=best_val_loss,
            epochs_without_improvement=epochs_without_improvement,
            history=history
        )

        save_json(training_state_path, {
            "last_completed_epoch": epoch,
            "next_start_epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "epochs_without_improvement": epochs_without_improvement,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "epochs_per_run": EPOCHS_PER_RUN,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "last_train_loss": train_loss,
            "last_val_loss": val_loss,
            "last_improvement": improvement,
            "last_generalization_gap": generalization_gap,
            "last_epoch_time_min": epoch_time_min
        })

    if epochs_without_improvement >= PATIENCE:
        print(f"\n🛑 Early stopping triggered at epoch {epoch + 1}")
        print(f"Best val_loss = {best_val_loss:.4f}")
        break
        
# FINAL SUMMARY
print("\n[8/8] Training session complete.")
print("Saved files:")
print(" -", last_ckpt_path)
print(" -", best_ckpt_path)
print(" -", history_path)
print(" -", training_state_path)

# optional: show final history lengths
if os.path.exists(history_path):
    with open(history_path, "r", encoding="utf-8") as f:
        hist = json.load(f)
    print("History lengths:",
          "train =", len(hist.get("train_loss", [])),
          "| val =", len(hist.get("val_loss", [])))


[7/8] Starting training ...

Epoch 15


Training:   0%|          | 0/1968 [00:00<?, ?it/s]

KeyboardInterrupt: 